# qnmlab quick start

Measure a black hole's ringing frequency with a quantum algorithm, in five steps.
Runs on a laptop in about a minute. On Colab, run `!pip install qnmlab matplotlib` first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qnmlab as ql
print("qnmlab", ql.__version__)

## 1. Choose a black hole and get the classical reference

`Schwarzschild` and `KazakovSolodukhin` are validated against published frequencies;
`ReissnerNordstrom` and `Hayward` are experimental.

In [ ]:
bh = ql.KazakovSolodukhin(a=0.5)
ref = ql.reference_modes(bh, ell=2, n_modes=2)
for n, w in enumerate(ref):
    print(f"overtone {n}:  M omega = {w:.6f}")

## 2. The operator a quantum processor reads

On hyperboloidal slices every mode is an eigenvalue of one matrix. With 3 qubits it is
8 x 8; more qubits make its eigenvalues more accurate.

In [ ]:
for q in (3, 4, 5):
    op = ql.readout_operator(bh, qubits=q)
    print(f"{q} qubits: fundamental {op.eigenvalues[0]:.6f}, error {abs(op.eigenvalues[0] - ref[0]):.1e}")

## 3. Read the frequency on a simulated noisy device

A quantum state is prepared close to the mode's eigenvector, then the frequency is read
from two measured averages: `Re ω = −⟨H₂⟩`, `Im ω = ⟨H₁⟩`.
The noise model reproduces the bias seen on the IQM Emerald processor.

In [ ]:
noise = ql.NoiseModel.emerald_random()
results = {shots: ql.measure_mode(bh, qubits=3, noise=noise, shots=shots, seed=s)
           for s, shots in enumerate((200, 800, 3200))}
for shots, r in results.items():
    print(f"{shots:5d} shots/circuit: omega = {r.omega:.4f}, error {r.meta['error_vs_reference']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(ref.real, ref.imag, "*", ms=14, label="reference modes")
for shots, r in results.items():
    ax.plot(r.omega.real, r.omega.imag, "o", label=f"noisy readout, {shots} shots")
ax.set_xlabel("Re Mω (pitch)"); ax.set_ylabel("Im Mω (decay)"); ax.legend(fontsize=8)
plt.show()

## 4. The real hardware data

The raw counts of the 2026 IQM Emerald run ship with the package. Re-analyse them:

In [ ]:
w_hw = ql.benchmarks.reanalyse_emerald_2026()
print("measured on IQM Emerald:", w_hw)
print(ql.benchmarks.compare(w_hw, ql.Schwarzschild()))

## 5. Find modes with no starting guess

`find_modes` scans a region of the frequency plane and verifies each candidate with the
readout. With `verify=False` it returns the shortlist in seconds; full verification at 4
qubits takes about 40 minutes.

In [ ]:
res = ql.find_modes(ql.Schwarzschild(), qubits=4, verify=False)
leads = np.array([c.lead for c in res.candidates])
print(f"{len(leads)} candidates for {res.shots:.1e} measurement shots")
print("closest to the fundamental:", leads[np.argmin(abs(leads - ql.reference_modes(ql.Schwarzschild())[0]))])